In [11]:
import numpy as np
import torch
from ase.io import read, write
from dscribe.descriptors import SOAP
from copy import deepcopy
from tqdm import trange

# --------------------------
# Paths
# --------------------------
ref_path = "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/meci/benzene_main_meci/Type 1.xyz"
target_path = "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/raw_geometries/meci/benzene_main_meci/Type 2.xyz"

# --------------------------
# Load geometries
# --------------------------
ref_atoms = read(ref_path)
target_atoms = read(target_path)

# --------------------------
# SOAP descriptor
# --------------------------
def generate_SOAP(atoms, r_cut=5.0, n_max=8, l_max=6, average="inner"):
    species = list(set(atoms.get_chemical_symbols()))
    soap = SOAP(
        species=species,
        periodic=False,
        r_cut=r_cut,
        n_max=n_max,
        l_max=l_max,
        average=average,
        sparse=False,
    )
    return soap.create([atoms]).flatten()

# Reference SOAP vector
soap_ref = generate_SOAP(ref_atoms)

# --------------------------
# Finite difference gradient
# --------------------------
def finite_diff_gradient(geometry, target_feature, eps=1e-4):
    """Compute numerical gradient of the squared distance between SOAP vectors."""
    grad = np.zeros_like(geometry)
    for i in range(geometry.shape[0]):
        for j in range(3):
            geom_plus = geometry.copy()
            geom_minus = geometry.copy()
            geom_plus[i, j] += eps
            geom_minus[i, j] -= eps
            
            # Set positions
            atoms_plus = deepcopy(target_atoms)
            atoms_plus.set_positions(geom_plus)
            atoms_minus = deepcopy(target_atoms)
            atoms_minus.set_positions(geom_minus)
            
            # SOAP features
            f_plus = generate_SOAP(atoms_plus)
            f_minus = generate_SOAP(atoms_minus)
            
            # Central difference
            grad[i, j] = (np.sum((f_plus - target_feature)**2) - np.sum((f_minus - target_feature)**2)) / (2 * eps)
    return grad

# --------------------------
# Optimization
# --------------------------
device = torch.device("cpu")  # or "cuda" if available
positions = torch.tensor(target_atoms.get_positions(), dtype=torch.float32, requires_grad=True, device=device)
optimizer = torch.optim.Adam([positions], lr=0.01)
n_steps = 2500

# Trajectory file
traj_file = "trajectory.xyz"
write(traj_file, target_atoms)  # Save initial geometry

for step in trange(n_steps):
    optimizer.zero_grad()
    
    # Compute finite difference gradient
    grad_np = finite_diff_gradient(positions.detach().cpu().numpy(), soap_ref)
    positions.grad = torch.tensor(grad_np, dtype=torch.float32)
    
    # Take optimizer step
    optimizer.step()
    
    # Save current geometry to trajectory
    current_atoms = deepcopy(target_atoms)
    current_atoms.set_positions(positions.detach().cpu().numpy())
    write(traj_file, current_atoms, append=True)
    
    # Compute loss
    soap_current = generate_SOAP(current_atoms)
    loss = np.sum((soap_current - soap_ref)**2)
    
    if step % 5 == 0:
        print(f"Step {step}, Loss: {loss:.6f}")

# --------------------------
# Save final optimized geometry
# --------------------------
optimized_atoms = deepcopy(target_atoms)
optimized_atoms.set_positions(positions.detach().cpu().numpy())
optimized_atoms.write("reconstructed_geometry.xyz")
print("Trajectory saved to", traj_file)
print("Final reconstructed geometry saved to reconstructed_geometry.xyz")


  0%|          | 7/2500 [00:00<01:24, 29.49it/s]

Step 0, Loss: 111.437140
Step 5, Loss: 79.333043


  1%|          | 15/2500 [00:00<01:21, 30.39it/s]

Step 10, Loss: 47.758529
Step 15, Loss: 26.653211


  1%|          | 26/2500 [00:00<01:23, 29.79it/s]

...

Step 2480, Loss: 0.016807
Step 2485, Loss: 0.016751


100%|█████████▉| 2497/2500 [01:22<00:00, 31.24it/s]

Step 2490, Loss: 0.016695
Step 2495, Loss: 0.016640


100%|██████████| 2500/2500 [01:22<00:00, 30.39it/s]

Trajectory saved to trajectory.xyz
Final reconstructed geometry saved to reconstructed_geometry.xyz
